# DL Training 

### Training a Neural Network Loop
1. create a model
2. Choose a loss function
3. define a dataset
4. set an optimizer
5. Run a traning loop:
   - Calculate Loss ( Forward pass)
   - Compute Gradients ( Backpropagation)
   - Updating model parameters

In [1]:
import os

os.getcwd()

'E:\\Files\\Jobs - 2026\\Models\\notebooks\\AI_ENGINEERING'

In [13]:
# Import nessasory Libs
import torch
import torch.nn as nn
from collections import OrderedDict
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch.optim as optim
from torch.utils.data import TensorDataset,DataLoader

torch.manual_seed(42)

# Numeric Output Dataset neural network Implemetation

In [26]:
# Importing the processed Dataset (unchanged)
X_test = pd.read_csv(r'..\..\data\Processed\X_test.csv')
X_train = pd.read_csv(r'..\..\data\Processed\X_train.csv')
y_test = pd.read_csv(r'..\..\data\Processed\y_test.csv')
y_train = pd.read_csv(r'..\..\data\Processed\y_train.csv')
X_test = X_test.drop(columns=['Country'])
X_train = X_train.drop(columns=['Country'])
print(f"Training Set   : {X_train.shape}, {y_train.shape}")
print(f"Test Set       : {X_test.shape}, {y_test.shape}")

# y comes in as (n, 1) DataFrames — squeeze to 1D before anything else
y_train = y_train.iloc[:, 0]
y_test = y_test.iloc[:, 0]

# Carve a validation set out of the training data ONLY — X_test/y_test stay untouched
X_train_fit, X_val, y_train_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42
)

# Scale features — fit only on the training fit split
x_scaler = StandardScaler()
X_train_scaled = x_scaler.fit_transform(X_train_fit)
X_val_scaled = x_scaler.transform(X_val)
X_test_scaled = x_scaler.transform(X_test)

# Scale target the same way
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train_fit.to_numpy().reshape(-1, 1)).flatten()
y_val_scaled = y_scaler.transform(y_val.to_numpy().reshape(-1, 1)).flatten()
y_test_scaled = y_scaler.transform(y_test.to_numpy().reshape(-1, 1)).flatten()

train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train_scaled, dtype=torch.float32),
        torch.tensor(y_train_scaled, dtype=torch.float32).unsqueeze(1)
    ),
    batch_size=32, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_val_scaled, dtype=torch.float32),
        torch.tensor(y_val_scaled, dtype=torch.float32).unsqueeze(1)
    ),
    batch_size=32, shuffle=False
)
test_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_test_scaled, dtype=torch.float32),
        torch.tensor(y_test_scaled, dtype=torch.float32).unsqueeze(1)
    ),
    batch_size=32, shuffle=False
)

model = nn.Sequential(OrderedDict([
    ("input", nn.Linear(16, 32)),
    ("bn1", nn.BatchNorm1d(32)),
    ("relu1", nn.ReLU()),
    ("l1", nn.Linear(32, 16)),
    ("bn2", nn.BatchNorm1d(16)),
    ("relu2", nn.ReLU()),
    ("output", nn.Linear(16, 1))
]))
print(model)

criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.89)

n_epochs = 121
for epoch in range(n_epochs):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            pred = model(xb)
            val_loss += criterion(pred, yb).item() * xb.size(0)
    val_loss /= len(val_loader.dataset)

    if epoch % 10 == 0 or epoch == n_epochs - 1:
        print(f"Epoch {epoch:3d} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f}")

Training Set   : (2342, 16), (2342, 1)
Test Set       : (586, 16), (586, 1)
Sequential(
  (input): Linear(in_features=16, out_features=32, bias=True)
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu1): ReLU()
  (l1): Linear(in_features=32, out_features=16, bias=True)
  (bn2): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu2): ReLU()
  (output): Linear(in_features=16, out_features=1, bias=True)
)
Epoch   0 | train_loss: 0.3203 | val_loss: 0.1841
Epoch  10 | train_loss: 0.1161 | val_loss: 0.0956
Epoch  20 | train_loss: 0.1008 | val_loss: 0.0870
Epoch  30 | train_loss: 0.1100 | val_loss: 0.1025
Epoch  40 | train_loss: 0.1009 | val_loss: 0.0789
Epoch  50 | train_loss: 0.0869 | val_loss: 0.0760
Epoch  60 | train_loss: 0.0938 | val_loss: 0.0794
Epoch  70 | train_loss: 0.0892 | val_loss: 0.0766
Epoch  80 | train_loss: 0.0715 | val_loss: 0.0721
Epoch  90 | train_loss: 0.0824 | val_loss

In [27]:
model.eval()
test_loss = 0.0
preds_list, actuals_list = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        pred = model(xb)
        test_loss += criterion(pred, yb).item() * xb.size(0)
        preds_list.append(pred)
        actuals_list.append(yb)
test_loss /= len(test_loader.dataset)
print(f"Test MSE (scaled): {test_loss:.4f}")

preds = torch.cat(preds_list).numpy()
actuals = torch.cat(actuals_list).numpy()
preds_real = y_scaler.inverse_transform(preds)
actuals_real = y_scaler.inverse_transform(actuals)

import numpy as np
rmse_years = np.sqrt(np.mean((preds_real - actuals_real) ** 2))
mae_years = np.mean(np.abs(preds_real - actuals_real))
print(f"Test RMSE: {rmse_years:.2f} years")
print(f"Test MAE : {mae_years:.2f} years")

Test MSE (scaled): 0.0616
Test RMSE: 2.39 years
Test MAE : 1.75 years


# Binary Class Output Dataset neural network Implemetation

In [ ]:
path = "../../data/Processed/phishing_websites_dataset"
X = pd.read_csv(path + "/X.csv")
y = pd.read_csv(path + "/y.csv")
print(X.shape)
print(y.shape)
print(y.value_counts(normalize = True))

In [ ]:
X = X.to_numpy()
y = y.to_numpy()
print(X)
print(y)

In [ ]:
X_train , X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.30, random_state = 143
)

In [ ]:
batch_size = 32
train_loader = DataLoader(TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).long()), batch_size = batch_size, shuffle = True) 
test_loader  = DataLoader(TensorDataset(torch.tensor(X_test).float(), torch.tensor(y_test).long()), batch_size = batch_size, shuffle = False)

In [ ]:
# To view gow the dataloder iterativly inputs the data 
for batch_inputs, batch_labels in train_loader:
    print('batch_inputs', batch_inputs)
    print( 'batch_labels', batch_labels)